In [1]:
# -*- coding: utf-8 -*-
"""
Created on Thu Sep  3 11:04:24 2020

@author: ddy17
"""
import warnings
import sys
import requests
import os
import pandas as pd
from pandas.core.common import SettingWithCopyWarning
import io
import UnumEmail
import paramiko
from datetime import date, timedelta


In [2]:
def LeaveReturn_url_generator(start_dt, end_dt):
    
    url = 'https://services1.myworkday.com/ccx/service/customreport2/unum/G3006/WS_Leave_Return_Completed_in_Range?Return_Completed_End={0}-07%3A00&Return_Completed_Start={1}-08%3A00&format=csv'.format(end_dt, start_dt)
    
    return url

def Qualtrics_url_generator():
    
    url = 'https://services1.myworkday.com/ccx/service/customreport2/unum/C2HXT/Qualtrics_Survey_Data_File?Supervisory_Organization%21WID=fad8c8a44c4510541494e34161d00311&Include_Subordinate_Organizations=1&format=csv'
    
    return url

def wd_report_pull(url, username, password):
    
    with requests.Session() as s:
        download = s.get(url, auth=(username, password))
        decoded_content = download.content.decode('utf-8')
        
    return pd.read_csv(io.StringIO(decoded_content))

def sftp_transfer(username, password, myHostname):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(myHostname, username = username, password = password)
    sftp = ssh.open_sftp()
    sftp.chdir('/Home/unumqualtrics/PeopleAnalytics/Workday/Out')
    
    try:
        sftp.remove('/Home/unumqualtrics/PeopleAnalytics/Workday/Out/LEAVE.csv')
        sftp.put('C:\\Users\\DDY17\\Anaconda3\\Anaconda3\\envs\\p35env\\notebooks\\LEAVE.csv', "/Home/unumqualtrics/PeopleAnalytics/Workday/Out/LEAVE.csv")

    except Exception:
        sftp.put('C:\\Users\\DDY17\\Anaconda3\\Anaconda3\\envs\\p35env\\notebooks\\LEAVE.csv', "/Home/unumqualtrics/PeopleAnalytics/Workday/Out/LEAVE.csv")
    
    return sftp.close()

In [3]:
def main():
    
    #end_dt = (date.today()).strftime('%Y-%m-%d') + '-07'
    #start_dt = (date.today() - timedelta(6)).strftime('%Y-%m-%d') + '-08'
    start_dt = '2021-03-25'
    end_dt = '2021-03-31'
        
    username = os.environ.get('WDUSER') #'BUF20' #os.environ.get('WDUSER')
    password = os.environ.get('WDPASSWORD') #'Dina4166!@' #
    
    myHostname = os.environ.get('SFTP_HOST') # #os.environ.get('SFTP_HOST')
    myUsername = os.environ.get('SFTP_USER') # #os.environ.get('SFTP_USER')
    myPassword = os.environ.get('SFTP_PASSWORD') #'' #os.environ.get('SFTP_PASSWORD')
    
    sender = 'do_not_reply@estevan.com'
    recipients = ['cwymer@unum.com']
    subject = 'Job Completed'
    body = 'Job # Leave Survey Completed!'
    
    
    try:
        print('pulling reports now....')
        leave_df = wd_report_pull(LeaveReturn_url_generator(start_dt, end_dt), username, password)
        print('leave report pulled...')
        qual_df = wd_report_pull(Qualtrics_url_generator(), username, password)
        print('Reports Pulled!')
        
        merged_df = qual_df.merge(leave_df[['Unique_Identifier','First_Day_Back']], left_on = 'Employee_ID', right_on = 'Unique_Identifier')
        deduped = merged_df.drop_duplicates()
        
        #deduped['Leave_Launch_Date'] = (date.today()).strftime('%m-%d-%Y')
        deduped['Leave_Launch_Date'] = '03-31-2021'
    
        deduped.to_csv('C:\\Users\\DDY17\\Anaconda3\\Anaconda3\\envs\\p35env\\notebooks\\LEAVE.csv', index=False)
        print('CSV Created!')
    except Exception as e:
        print('\nWD Report pull failed!')
        subject = 'Job Failed!'
        body = print('Job Failed!')
        UnumEmail.send_email(sender, recipients, subject, body, file_attached = False)
        print(str(e))
        sys.exit()
    
    try:
        print('Begin File Transfer!')
        sftp_transfer(myUsername, myPassword, myHostname)
    except Exception as e:
        print('\nFile transfer incomplete')
        print(str(e))
        subject = 'Job Failed!'
        body = 'Job Failed!'
        UnumEmail.send_email(sender, recipients, subject, body, file_attached = True, attachment = 'LEAVE.csv', attach_path = './LEAVE.csv')
        sys.exit()
        
    os.remove('C:\\Users\\DDY17\\Anaconda3\\Anaconda3\\envs\\p35env\\notebooks\\LEAVE.csv')
    print('file transfer complete!')

    UnumEmail.send_email(sender, recipients, subject, body, file_attached = False)
    print('email sent!')
    ()

if __name__ == '__main__':
        main()


pulling reports now....
leave report pulled...
Reports Pulled!
CSV Created!
Begin File Transfer!


<ipython-input-3-ad0d69d12f55>:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  deduped['Leave_Launch_Date'] = '03-31-2021'


file transfer complete!
email sent!


Socket exception: An existing connection was forcibly closed by the remote host (10054)
